# D011 | 따릉이 공공데이터 결합 실증 재현

원본은 사용자가 `data_raw/`에 보관합니다. 본 노트북에는 임의 생성한 데이터나 실행 결과를 삽입하지 않았습니다.

In [ ]:
from pathlib import Path
import pandas as pd
raw_dir = Path("data_raw")
files = list(raw_dir.glob("*.xlsx")) + list(raw_dir.glob("*.csv"))
print([(p.name,p.stat().st_size) for p in files])
assert len(list(raw_dir.glob("*.xlsx"))) == 1
assert len(list(raw_dir.glob("*.csv"))) == 1

## 자료 로딩
원본 XLSX의 머리글은 다중 행이므로 대여소 번호가 있는 행만 선택합니다. 기본 `read_excel`에서 다중 머리글 때문에 결측이 나타나는 것은 즉시 품질 오류라고 볼 수 없습니다.

In [ ]:
base = pd.read_excel(next(raw_dir.glob("*.xlsx")))
usage = pd.read_csv(next(raw_dir.glob("*.csv")), encoding="utf-8-sig")
print(base.shape,usage.shape)
print(usage.columns.tolist())
display(base.head(4), usage.head(4))

## 식별자 정규화와 결합
대여소 번호 추출 성공과 기준정보 유일성을 먼저 검증합니다.

In [ ]:
base["station_id"] = pd.to_numeric(base.iloc[:,0], errors="coerce").astype("Int64")
base_clean = base.loc[base["station_id"].notna()].copy()
base_clean["기준대여소명"] = base_clean.iloc[:,1]
usage["station_id"] = pd.to_numeric(usage["대여소명"].astype(str).str.extract(r"^\s*(\d+)\s*\.",expand=False), errors="coerce").astype("Int64")
print("base unique duplicate:",base_clean["station_id"].duplicated().sum())
print("usage parse missing:",usage["station_id"].isna().sum())
assert base_clean["station_id"].duplicated().sum() == 0
joined = usage.merge(base_clean[["station_id","기준대여소명"]],on="station_id",how="left",indicator=True,validate="many_to_one")
print(joined["_merge"].value_counts())

## 미연결 규모와 영향
단순 행 누락률 외에 실제 이용량 비율을 계산합니다. 월별 기준정보가 달라지므로 원인 판단은 유보합니다.

In [ ]:
unmatched=joined.loc[joined["_merge"]=="left_only"].copy()
monthly=joined.groupby("기준년월").agg(전체기록=("_merge","size"),미연결=("_merge",lambda s:(s=="left_only").sum())).reset_index()
monthly["미연결률(%)"]=(100*monthly["미연결"]/monthly["전체기록"]).round(3)
display(monthly)
print("unique unmatched",unmatched["station_id"].nunique())
for metric in ["대여건수","반납건수"]:
  values=pd.to_numeric(joined[metric].astype(str).str.replace(",","",regex=False),errors="coerce")
  affected=values[joined["_merge"]=="left_only"].sum();total=values.sum()
  print(metric, "total",total,"unmatched",affected,"percent",round(100*affected/total,3))

## 저장과 검증 결과 해석
이 결과는 데이터 결합 품질의 재현이며 실제 미연결 원인을 확정하거나 정부의 정책 효과를 검증하는 것이 아닙니다.

In [ ]:
out=Path("outputs");out.mkdir(exist_ok=True)
monthly.to_csv(out/"monthly_quality.csv",index=False,encoding="utf-8-sig")
unmatched.to_csv(out/"unmatched_usage.csv",index=False,encoding="utf-8-sig")
print("saved outputs")